In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import os; os.chdir('/home/jovyan/work/ufc/')

In [4]:
import pandas as pd
import joblib

from ruamel.yaml import YAML
import re
import numpy as np

conf = YAML().load(open('params.yaml'))

In [96]:
import pandas as pd
from ruamel.yaml import YAML
import numpy as np
import click
import time 

import os
import re

from functools import partial
from fastcore.basics import chunked

import sys
sys.path.append('.')

from src.stat_funcs import get_stat_feat, last_el
from src.constants import ActivityLogger

from pandarallel import pandarallel

pandarallel.initialize(progress_bar=False, nb_workers=os.cpu_count())

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


# Остальное

## диагностика памяти

In [ ]:
import os, psutil

mem_total = psutil.virtual_memory().total/(1024**3)
mem_av = psutil.virtual_memory().available/(1024**3)
mem_us = psutil.virtual_memory().used/(1024**3)
mem_pus = psutil.virtual_memory().percent
print(mem_total, mem_av, mem_us, mem_pus)

In [18]:
fights_df.groupby('chunk').parallel_apply(visible_get_stat_feat_chunk)

/tmp/ipykernel_602057/254893189.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('Fighter').apply(lambda x: fn(x, aggs = aggs, cust_aggs=cust_aggs, cols = stat_cols,


kd_stat_rol_sum  kd_dam_stat_rol_sum  \
chunk Fighter                                                           
0.0   Khabib Nurmagomedov 10069              NaN                  NaN   
                          10236              NaN                  NaN   
                          10395              NaN                  NaN   
                          2288          0.604157                  0.0   
                          10655         0.521739                  0.0   
                          10914         0.521739                  0.0   
                          3618          0.000000                  0.0   
                          3880          0.000000                  0.0   
                          4436          0.000000                  0.0   
                          4552          0.000000                  0.0   
                          4783          0.000000                  0.0   
                          5263          0.000000                  0.0   
                          5798          0.000000                  0.0   
      Royce Gracie        2                  NaN                  NaN   
                          5                  NaN                  NaN   
                          6                  NaN                  NaN   
                          10            0.000000                  0.0   
                          12            0.000000                  0.0   
                          15            0.000000                  0.0   
                          16            0.000000                  0.0   
                          24            0.000000                  0.0   
                          31            0.000000                  0.0   
                          32            0.000000                  0.0   
                          36            0.000000                  0.0   
                          8299          0.000000                  0.0   
                          8805          0.000000                  0.0   

                                 sub_att_stat_rol_sum  \
chunk Fighter                                           
0.0   Khabib Nurmagomedov 10069                   NaN   
                          10236                   NaN   
                          10395                   NaN   
                          2288               0.082418   
                          10655              0.133333   
                          10914              0.133333   
                          3618               0.133333   
                          3880               0.000000   
                          4436               0.079893   
                          4552               0.079893   
                          4783               0.119893   
                          5263               0.095402   
                          5798               0.260691   
      Royce Gracie        2                       NaN   
                          5                       NaN   
                          6                       NaN   
                          10                 1.629555   
                          12                 2.525077   
                          15                 2.607495   
                          16                 1.554863   
                          24                 1.048951   
                          31                 0.603896   
                          32                 0.865905   
                          36                 0.665968   
                          8299               0.993851   
                          8805               0.731842   

                                 sub_att_dam_stat_rol_sum  rev_stat_rol_sum  \
chunk Fighter                                                                 
0.0   Khabib Nurmagomedov 10069                       NaN               NaN   
                          10236                       NaN               NaN   
                          10395                       NaN               NaN

## поиск границы для f1

In [ ]:
from sklearn.metrics import f1_score
thresh_l = []
f1_l = []
for thresh in np.linspace(0,1,100):
    df['y_p'] = (df['score1']>=thresh).astype(int)
    thresh_l.append(thresh)
    f1_l.append(f1_score(df.loc[df.split=='tr', 'target'], df.loc[df.split=='tr', 'y_p']))
f1_df = pd.DataFrame({'thresh': thresh_l, 'f1': f1_l})

f1_df = f1_df.sort_values(by='f1', ascending=False)
display(f1_df.head(5))
thresh = f1_df['thresh'].iloc[0]

display(f1_df[f1_df['thresh']>0.5].head())